# ⚽ Goalkeeper Training - Professional Edition
A Pygame-based goalkeeper training simulation with AI positioning, threat detection, and shot prediction.

**Controls:**
- `WASD` / Arrow Keys — Move attacker
- `SPACE` — Shoot (aims toward mouse cursor)
- `R` — Reset ball to attacker
- `Mouse` — Aim direction

## 1. Imports & Game Constants

In [8]:
import pygame
import math
import random
import numpy as np
from enum import Enum

# --- Field Dimensions ---
FIELD_WIDTH = 1200
FIELD_HEIGHT = 800

# --- Entity Sizes ---
GOALKEEPER_SIZE = 30
ATTACKER_SIZE = 30
BALL_RADIUS = 10

# --- Goal & Box Dimensions ---
GOAL_WIDTH = 180
GOAL_HEIGHT = 120
PENALTY_AREA_WIDTH = 165
PENALTY_AREA_HEIGHT = 400
SIX_YARD_WIDTH = 55
SIX_YARD_HEIGHT = 200

# --- Physics & Movement ---
ATTACKER_SPEED = 4
CENTER_CIRCLE_RADIUS = 90
BALL_SPEED = 8
KICK_POWER = 12
BALL_FRICTION = 0.97

# --- Colors ---
FIELD_GREEN       = (34, 139, 34)
LINE_WHITE        = (255, 255, 255)
GOAL_WHITE        = (240, 240, 240)
GOALKEEPER_BLUE   = (30, 144, 255)
ATTACKER_RED      = (220, 20, 60)
BALL_WHITE        = (255, 255, 255)
PREDICTION_PURPLE = (138, 43, 226)
GRASS_STRIPE_LIGHT = (40, 150, 40)
GRASS_STRIPE_DARK  = (30, 130, 30)

## 2. Enums — Goalkeeper State & Dive Direction

In [9]:
class GoalkeeperState(Enum):
    IDLE       = 0
    TRACKING   = 1
    READY      = 2
    DIVING     = 3
    RECOVERING = 4

class DiveDirection(Enum):
    NONE  = 0
    LEFT  = 1
    RIGHT = 2
    DOWN  = 3

## 3. Pygame Initialisation

In [10]:
pygame.init()
screen = pygame.display.set_mode((FIELD_WIDTH, FIELD_HEIGHT))
pygame.display.set_caption("Goalkeeper Training - Professional Edition")
clock = pygame.time.Clock()

font       = pygame.font.Font(None, 24)
big_font   = pygame.font.Font(None, 36)
small_font = pygame.font.Font(None, 18)

## 4. Pitch Class
Handles all field rendering: grass stripes, lines, boxes, arcs, corner flags.

In [11]:
class Pitch:
    def __init__(self):
        self.width    = FIELD_WIDTH
        self.height   = FIELD_HEIGHT
        self.center_x = FIELD_WIDTH  // 2
        self.center_y = FIELD_HEIGHT // 2

        self.goal_x    = FIELD_WIDTH - 5
        self.goal_y    = (FIELD_HEIGHT - GOAL_HEIGHT) // 2

        self.penalty_x = FIELD_WIDTH - PENALTY_AREA_WIDTH
        self.penalty_y = (FIELD_HEIGHT - PENALTY_AREA_HEIGHT) // 2

        self.six_yard_x = FIELD_WIDTH - SIX_YARD_WIDTH
        self.six_yard_y = (FIELD_HEIGHT - SIX_YARD_HEIGHT) // 2

        self.stripe_width = 60
        self.num_stripes  = FIELD_WIDTH // self.stripe_width + 1

    def draw_grass_stripes(self, surface):
        """Draw alternating light/dark grass stripes."""
        for i in range(self.num_stripes):
            x     = i * self.stripe_width
            color = GRASS_STRIPE_LIGHT if i % 2 == 0 else GRASS_STRIPE_DARK
            pygame.draw.rect(surface, color, (x, 0, self.stripe_width, FIELD_HEIGHT))

    def draw(self, surface):
        surface.fill(FIELD_GREEN)
        self.draw_grass_stripes(surface)

        # Outer boundary
        pygame.draw.rect(surface, LINE_WHITE, (0, 0, FIELD_WIDTH, FIELD_HEIGHT), 4)
        pygame.draw.rect(surface, LINE_WHITE, (3, 3, FIELD_WIDTH-6, FIELD_HEIGHT-6), 2)

        # Centre line
        pygame.draw.line(surface, (200,200,200), (self.center_x+1,1), (self.center_x+1,FIELD_HEIGHT-1), 2)
        pygame.draw.line(surface, LINE_WHITE,    (self.center_x,0),   (self.center_x,FIELD_HEIGHT), 4)

        # Centre circle & spot
        pygame.draw.circle(surface, (200,200,200), (self.center_x+1,self.center_y+1), CENTER_CIRCLE_RADIUS, 3)
        pygame.draw.circle(surface, LINE_WHITE,    (self.center_x,self.center_y),     CENTER_CIRCLE_RADIUS, 4)
        pygame.draw.circle(surface, (255,255,200), (self.center_x,self.center_y), 8)
        pygame.draw.circle(surface, LINE_WHITE,    (self.center_x,self.center_y), 6)

        # Penalty & six-yard boxes (right & left)
        pygame.draw.rect(surface, LINE_WHITE, (self.penalty_x,  self.penalty_y,  PENALTY_AREA_WIDTH, PENALTY_AREA_HEIGHT), 4)
        pygame.draw.rect(surface, LINE_WHITE, (self.six_yard_x, self.six_yard_y, SIX_YARD_WIDTH,     SIX_YARD_HEIGHT),     4)
        pygame.draw.rect(surface, LINE_WHITE, (0, self.penalty_y,  PENALTY_AREA_WIDTH, PENALTY_AREA_HEIGHT), 4)
        pygame.draw.rect(surface, LINE_WHITE, (0, self.six_yard_y, SIX_YARD_WIDTH,     SIX_YARD_HEIGHT),     4)

        # Penalty spots
        psx = FIELD_WIDTH - 110
        for px in [psx, 110]:
            pygame.draw.circle(surface, (255,255,200), (px, self.center_y), 7)
            pygame.draw.circle(surface, LINE_WHITE,    (px, self.center_y), 6)

        # Penalty arcs
        pygame.draw.arc(surface, LINE_WHITE, pygame.Rect(psx-90, self.center_y-90, 180,180), math.pi/2, 3*math.pi/2, 4)
        pygame.draw.arc(surface, LINE_WHITE, pygame.Rect(110-90, self.center_y-90, 180,180), -math.pi/2, math.pi/2,  4)

        # Corner arcs
        cr = 12
        pygame.draw.arc(surface, LINE_WHITE, (FIELD_WIDTH-cr, -cr,               cr*2,cr*2), math.pi,     3*math.pi/2, 4)
        pygame.draw.arc(surface, LINE_WHITE, (FIELD_WIDTH-cr, FIELD_HEIGHT-cr,   cr*2,cr*2), math.pi/2,   math.pi,     4)
        pygame.draw.arc(surface, LINE_WHITE, (-cr,            -cr,               cr*2,cr*2), 0,           math.pi/2,   4)
        pygame.draw.arc(surface, LINE_WHITE, (-cr,            FIELD_HEIGHT-cr,   cr*2,cr*2), 3*math.pi/2, 2*math.pi,   4)

        # Corner flags
        for fx, fy in [(FIELD_WIDTH-8,8),(FIELD_WIDTH-8,FIELD_HEIGHT-8),(8,8),(8,FIELD_HEIGHT-8)]:
            self.draw_corner_flag(surface, fx, fy)

    def draw_corner_flag(self, surface, x, y):
        pygame.draw.line(surface, (200,200,200), (x,y), (x,y-20), 2)
        pygame.draw.polygon(surface, ATTACKER_RED, [(x,y-20),(x+12,y-15),(x,y-10)])

## 5. Goal Class
Draws posts, crossbar, net, and detects whether the ball has crossed the goal line.

In [12]:
class Goal:
    def __init__(self, x, y):
        self.x      = x
        self.y      = y
        self.width  = 10
        self.height = GOAL_HEIGHT
        self.rect   = pygame.Rect(x-5, y, 15, GOAL_HEIGHT)

    def draw_net(self, surface):
        spacing = 8
        for i in range(0, self.height, spacing):
            pygame.draw.line(surface, (180,180,180), (self.x-35, self.y+i), (self.x, self.y+i), 1)
        for i in range(-35, 0, spacing):
            pygame.draw.line(surface, (180,180,180), (self.x+i, self.y), (self.x+i, self.y+self.height), 1)

    def draw(self, surface):
        self.draw_net(surface)

        # Posts
        pygame.draw.rect(surface, (220,220,220), (self.x-2, self.y-2, self.width+4, 10))
        pygame.draw.rect(surface, GOAL_WHITE,    (self.x,   self.y,   self.width,   8))
        pygame.draw.rect(surface, (220,220,220), (self.x-2, self.y+self.height-6, self.width+4, 10))
        pygame.draw.rect(surface, GOAL_WHITE,    (self.x,   self.y+self.height-8, self.width,   8))

        # Upright
        pygame.draw.rect(surface, (200,200,200), (self.x-2, self.y-2, 10, self.height+4))
        pygame.draw.rect(surface, GOAL_WHITE,    (self.x,   self.y,   8,  self.height))
        pygame.draw.rect(surface, (255,255,255), (self.x+1, self.y+1, 2,  6))

        # Goal line
        pygame.draw.line(surface, (200,200,200), (self.x-6, self.y), (self.x-6, self.y+self.height), 6)
        pygame.draw.line(surface, LINE_WHITE,    (self.x-5, self.y), (self.x-5, self.y+self.height), 5)

    def check_goal(self, ball):
        return (ball.x >= self.x - 5 and
                ball.y >= self.y and
                ball.y <= self.y + self.height)

## 6. Goalkeeper Class
AI-controlled goalkeeper with optimal positioning, shot prediction, and dive logic.

In [13]:
class Goalkeeper:
    def __init__(self, x, y):
        self.x = x;  self.y = y
        self.home_x = x;  self.home_y = y
        self.size   = GOALKEEPER_SIZE
        self.state  = GoalkeeperState.IDLE
        self.dive_direction = DiveDirection.NONE
        self.dive_timer     = 0
        self.dive_distance  = 50

        self.target_x       = x
        self.target_y       = y
        self.movement_speed = 2.5

        self.threat_level         = 0
        self.angle_coverage       = 0
        self.optimal_position     = (x, y)
        self.predicted_shot_angle = 0
        self.confidence           = 0

    # ── Positioning ──────────────────────────────────────────────────────────
    def calculate_optimal_position(self, attacker, ball):
        goal_center_x = FIELD_WIDTH - 25
        goal_center_y = FIELD_HEIGHT // 2

        threat_x = ball.x if abs(ball.dx)>1 or abs(ball.dy)>1 else attacker.x
        threat_y = ball.y if abs(ball.dx)>1 or abs(ball.dy)>1 else attacker.y

        top_post_y    = goal_center_y - GOAL_HEIGHT//2
        bottom_post_y = goal_center_y + GOAL_HEIGHT//2

        angle_top    = math.atan2(top_post_y    - threat_y, goal_center_x - threat_x)
        angle_bottom = math.atan2(bottom_post_y - threat_y, goal_center_x - threat_x)
        bisector     = (angle_top + angle_bottom) / 2

        dist = math.sqrt((threat_x-goal_center_x)**2 + (threat_y-goal_center_y)**2)
        pd   = min(45, max(20, dist/8))

        opt_x = goal_center_x - pd
        opt_y = goal_center_y - math.tan(bisector)*pd
        opt_y = max(goal_center_y-GOAL_HEIGHT//2+15, min(goal_center_y+GOAL_HEIGHT//2-15, opt_y))
        self.optimal_position = (opt_x, opt_y)

        ball_speed = math.sqrt(ball.dx**2 + ball.dy**2)
        if dist < 400:
            aw = abs(angle_top - angle_bottom)
            self.threat_level = min(1.0, ((400-dist)/400)*aw*(1+min(1.0,ball_speed/5)))
        else:
            self.threat_level = 0

    # ── Shot Prediction ───────────────────────────────────────────────────────
    def predict_shot_direction(self, ball, attacker):
        if abs(ball.dx)>2 or abs(ball.dy)>2:
            future_x = ball.x + ball.dx*20
            future_y = ball.y + ball.dy*20
            goal_cy  = FIELD_HEIGHT//2
            angle    = math.atan2(future_y-goal_cy, FIELD_WIDTH-future_x)
            self.predicted_shot_angle = angle
            self.confidence = min(1.0, math.sqrt(ball.dx**2+ball.dy**2)/10)
            return angle, self.confidence
        return 0, 0

    # ── Dive Decision ─────────────────────────────────────────────────────────
    def decide_dive(self):
        if self.confidence>0.6 and self.threat_level>0.8:
            goal_cy        = FIELD_HEIGHT//2
            impact_y = goal_cy + math.tan(self.predicted_shot_angle)*50
            if   impact_y < self.y - 20: return DiveDirection.LEFT
            elif impact_y > self.y + 20: return DiveDirection.RIGHT
            else:                        return DiveDirection.DOWN
        return DiveDirection.NONE

    # ── Main Update ──────────────────────────────────────────────────────────
    def update(self, attacker, ball, shoot_pressed):
        self.calculate_optimal_position(attacker, ball)
        self.predict_shot_direction(ball, attacker)

        if self.state in [GoalkeeperState.IDLE, GoalkeeperState.TRACKING, GoalkeeperState.READY]:
            if self.threat_level > 0.8:
                self.state = GoalkeeperState.READY
                if shoot_pressed or (abs(ball.dx)>3 or abs(ball.dy)>3):
                    dd = self.decide_dive()
                    if dd != DiveDirection.NONE:
                        self.dive_direction = dd
                        self.state      = GoalkeeperState.DIVING
                        self.dive_timer = 25
            elif self.threat_level > 0.4:
                self.state = GoalkeeperState.TRACKING
            else:
                self.state = GoalkeeperState.IDLE

            tx, ty = self.optimal_position
            dx = tx - self.x;  dy = ty - self.y
            d  = math.sqrt(dx*dx + dy*dy)
            if d > 3:
                spd = self.movement_speed*(0.5+self.threat_level*0.5)
                self.x += (dx/d)*spd;  self.y += (dy/d)*spd

        elif self.state == GoalkeeperState.DIVING:
            self.dive_timer -= 1
            if   self.dive_direction == DiveDirection.LEFT:  self.x-=4; self.y-=2
            elif self.dive_direction == DiveDirection.RIGHT: self.x+=4; self.y+=2
            elif self.dive_direction == DiveDirection.DOWN:  self.y+=3
            if self.dive_timer <= 0:
                self.state      = GoalkeeperState.RECOVERING
                self.dive_timer = 30

        elif self.state == GoalkeeperState.RECOVERING:
            self.dive_timer -= 1
            self.x += (self.home_x-self.x)*0.08
            self.y += (self.home_y-self.y)*0.08
            if self.dive_timer <= 0:
                self.state          = GoalkeeperState.IDLE
                self.dive_direction = DiveDirection.NONE
                self.x = self.home_x;  self.y = self.home_y

    # ── Save Detection ───────────────────────────────────────────────────────
    def check_save(self, ball):
        dist = math.sqrt((ball.x-self.x)**2 + (ball.y-self.y)**2)
        sr   = self.size//2 + ball.radius + 5
        if self.state == GoalkeeperState.DIVING:
            sr += 15
        return dist < sr

    # ── Drawing ──────────────────────────────────────────────────────────────
    def draw(self, surface):
        color_map = {
            GoalkeeperState.READY:      (255,100,100),
            GoalkeeperState.DIVING:     (255,50,50),
            GoalkeeperState.TRACKING:   (255,200,100),
            GoalkeeperState.RECOVERING: (200,200,255),
        }
        color = color_map.get(self.state, GOALKEEPER_BLUE)

        pygame.draw.circle(surface, (0,0,0), (int(self.x),int(self.y)), self.size//2+2)
        pygame.draw.circle(surface, color,   (int(self.x),int(self.y)), self.size//2)
        surface.blit(small_font.render("1",True,LINE_WHITE),
                     small_font.render("1",True,LINE_WHITE).get_rect(center=(int(self.x),int(self.y))))

        go = self.size//3
        pygame.draw.circle(surface, (255,255,100), (int(self.x-go),int(self.y)), 6)
        pygame.draw.circle(surface, (255,255,100), (int(self.x+go),int(self.y)), 6)
        pygame.draw.circle(surface, LINE_WHITE, (int(self.x),int(self.y)), self.size//2, 2)

        # Threat ring
        if self.threat_level > 0.1:
            tr = int(20+self.threat_level*20)
            tc = (255, int(255*(1-self.threat_level)), int(255*(1-self.threat_level)))
            pygame.draw.circle(surface, tc, (int(self.x),int(self.y)), tr, 2)

        # Optimal-position guide
        if self.state in [GoalkeeperState.IDLE,GoalkeeperState.TRACKING,GoalkeeperState.READY]:
            ox, oy = self.optimal_position
            if abs(ox-self.x)>3 or abs(oy-self.y)>3:
                pygame.draw.circle(surface, PREDICTION_PURPLE, (int(ox),int(oy)), 8, 2)
                pygame.draw.line(surface, PREDICTION_PURPLE, (self.x,self.y), (ox,oy), 1)

        # Prediction arrow
        if self.confidence > 0.3:
            al  = 40*self.confidence
            ex  = self.x + math.cos(self.predicted_shot_angle)*al
            ey  = self.y + math.sin(self.predicted_shot_angle)*al
            pygame.draw.line(surface, PREDICTION_PURPLE, (self.x,self.y), (ex,ey), 3)
            hs  = 8; aa = self.predicted_shot_angle
            pygame.draw.polygon(surface, PREDICTION_PURPLE, [
                (ex, ey),
                (ex-hs*math.cos(aa-0.5), ey-hs*math.sin(aa-0.5)),
                (ex-hs*math.cos(aa+0.5), ey-hs*math.sin(aa+0.5))
            ])

## 7. Attacker Class
Player-controlled entity that carries or releases the ball.

In [14]:
class Attacker:
    def __init__(self, x, y):
        self.x = x;  self.y = y
        self.size = ATTACKER_SIZE
        self.dx = 0;  self.dy = 0
        self.has_ball = True

    def update(self, keys):
        self.dx = 0;  self.dy = 0
        if keys[pygame.K_w] or keys[pygame.K_UP]:    self.dy = -ATTACKER_SPEED
        if keys[pygame.K_s] or keys[pygame.K_DOWN]:  self.dy =  ATTACKER_SPEED
        if keys[pygame.K_a] or keys[pygame.K_LEFT]:  self.dx = -ATTACKER_SPEED
        if keys[pygame.K_d] or keys[pygame.K_RIGHT]: self.dx =  ATTACKER_SPEED

        self.x = max(self.size//2, min(FIELD_WIDTH -self.size//2-50, self.x+self.dx))
        self.y = max(self.size//2, min(FIELD_HEIGHT-self.size//2,    self.y+self.dy))

    def draw(self, surface):
        color = ATTACKER_RED if self.has_ball else (150,150,150)
        pygame.draw.circle(surface, (0,0,0), (int(self.x),int(self.y)), self.size//2+2)
        pygame.draw.circle(surface, color,   (int(self.x),int(self.y)), self.size//2)
        surface.blit(small_font.render("7",True,LINE_WHITE),
                     small_font.render("7",True,LINE_WHITE).get_rect(center=(int(self.x),int(self.y))))
        pygame.draw.circle(surface, LINE_WHITE, (int(self.x),int(self.y)), self.size//2, 2)
        if abs(self.dx)>0.5 or abs(self.dy)>0.5:
            pygame.draw.line(surface, ATTACKER_RED,
                             (self.x,self.y), (self.x+self.dx*8, self.y+self.dy*8), 3)

## 8. Ball Class
Physics-based ball with motion trail, bounce, and shot mechanics.

In [15]:
class Ball:
    def __init__(self, x, y):
        self.x = x;  self.y = y
        self.dx = 0;  self.dy = 0
        self.radius  = BALL_RADIUS
        self.trail   = []
        self.is_shot = False

    def update(self, attacker):
        if not self.is_shot and attacker.has_ball:
            off = 20 if attacker.dx >= 0 else -20
            self.x = attacker.x + off
            self.y = attacker.y
        else:
            self.x += self.dx;  self.y += self.dy
            self.dx *= BALL_FRICTION;  self.dy *= BALL_FRICTION

            if abs(self.dx)>1 or abs(self.dy)>1:
                self.trail.append((self.x,self.y))
                if len(self.trail)>10: self.trail.pop(0)
            else:
                self.trail = []

            if self.x-self.radius<=0 or self.x+self.radius>=FIELD_WIDTH:
                self.dx *= -0.8
                self.x = max(self.radius, min(FIELD_WIDTH-self.radius, self.x))
            if self.y-self.radius<=0 or self.y+self.radius>=FIELD_HEIGHT:
                self.dy *= -0.8
                self.y = max(self.radius, min(FIELD_HEIGHT-self.radius, self.y))

            if abs(self.dx)<0.5 and abs(self.dy)<0.5:
                self.dx=0; self.dy=0; self.is_shot=False

    def shoot(self, target_x, target_y, attacker):
        if attacker.has_ball:
            dx = target_x-self.x;  dy = target_y-self.y
            d  = math.sqrt(dx*dx+dy*dy)
            if d>0:
                self.dx = (dx/d)*KICK_POWER
                self.dy = (dy/d)*KICK_POWER
                self.is_shot      = True
                attacker.has_ball = False

    def reset_to_attacker(self, attacker):
        self.x = attacker.x+20;  self.y = attacker.y
        self.dx=0; self.dy=0; self.is_shot=False; self.trail=[]
        attacker.has_ball = True

    def draw(self, surface):
        for i,(tx,ty) in enumerate(self.trail):
            a  = (i+1)/len(self.trail)
            tr = int(self.radius*a*0.7)
            cv = int(150+105*a)
            pygame.draw.circle(surface, (cv,cv,cv), (int(tx),int(ty)), tr)

        pygame.draw.ellipse(surface, (80,80,80),
                            (int(self.x-self.radius), int(self.y+self.radius-2), self.radius*2, 6))
        pygame.draw.circle(surface, (240,240,240), (int(self.x),int(self.y)), self.radius)
        pygame.draw.circle(surface, BALL_WHITE,    (int(self.x-2),int(self.y-2)), self.radius-2)
        pygame.draw.circle(surface, (0,0,0),       (int(self.x),int(self.y)), self.radius, 2)
        pygame.draw.circle(surface, (255,255,255), (int(self.x-3),int(self.y-3)), 3)

## 9. HUD / UI Helper
Draws stats panel, threat bar, and on-screen instructions.

In [16]:
def draw_ui(goalkeeper, attacker, ball, stats):
    # ── Stats panel ──────────────────────────────────────────────────────────
    ui_bg = pygame.Surface((300,200), pygame.SRCALPHA)
    ui_bg.fill((0,0,0,100))
    screen.blit(ui_bg, (5,5))

    screen.blit(font.render(f"GK State: {goalkeeper.state.name}", True, LINE_WHITE), (10,10))
    screen.blit(font.render(f"Threat: {goalkeeper.threat_level:.2f}",   True, LINE_WHITE), (10,35))
    screen.blit(font.render(f"Prediction: {goalkeeper.confidence:.2f}", True, LINE_WHITE), (10,60))
    screen.blit(font.render(f"Saves: {stats['saves']}",  True, (100,255,100)), (10, 90))
    screen.blit(font.render(f"Goals: {stats['goals']}",  True, (255,100,100)), (10,115))
    screen.blit(font.render(f"Shots: {stats['shots']}",  True, LINE_WHITE),    (10,140))
    if stats['shots'] > 0:
        sr = (stats['saves']/stats['shots'])*100
        screen.blit(font.render(f"Save Rate: {sr:.1f}%", True, (255,255,100)), (10,165))

    # ── Instructions ─────────────────────────────────────────────────────────
    inst_bg = pygame.Surface((FIELD_WIDTH-20,60), pygame.SRCALPHA)
    inst_bg.fill((0,0,0,120))
    screen.blit(inst_bg, (10,FIELD_HEIGHT-65))
    screen.blit(font.render("WASD/Arrows: Move | SPACE: Shoot | Mouse: Aim", True, LINE_WHITE),(15,FIELD_HEIGHT-55))
    screen.blit(font.render("R: Reset | Purple = Optimal position",          True, LINE_WHITE),(15,FIELD_HEIGHT-30))

    # ── Possession indicator ─────────────────────────────────────────────────
    poss_bg = pygame.Surface((260,30), pygame.SRCALPHA)
    poss_bg.fill((0,0,0,120))
    screen.blit(poss_bg, (FIELD_WIDTH-270,5))
    poss = 'Attacker' if attacker.has_ball else 'Free'
    screen.blit(font.render(f"Possession: {poss}", True, LINE_WHITE), (FIELD_WIDTH-260,5))

    # ── Threat-level bar ─────────────────────────────────────────────────────
    bw  = 200;  bh = 20
    bx  = FIELD_WIDTH - bw - 40;  by = 45
    bar_bg = pygame.Surface((bw+20,50), pygame.SRCALPHA)
    bar_bg.fill((0,0,0,120))
    screen.blit(bar_bg, (bx-10, by-25))
    screen.blit(font.render("Threat Level", True, LINE_WHITE), (bx, by-20))
    pygame.draw.rect(screen, (60,60,60),   (bx, by, bw, bh))
    tw = int(bw*goalkeeper.threat_level)
    tc = (255, int(255*(1-goalkeeper.threat_level)), 0)
    pygame.draw.rect(screen, tc,           (bx, by, tw, bh))
    pygame.draw.rect(screen, LINE_WHITE,   (bx, by, bw, bh), 2)

## 10. Game Initialisation
Create all game objects and reset stats.

In [17]:
pitch      = Pitch()
goal       = Goal(FIELD_WIDTH-10, (FIELD_HEIGHT-GOAL_HEIGHT)//2)
goalkeeper = Goalkeeper(FIELD_WIDTH-40, FIELD_HEIGHT//2)
attacker   = Attacker(300, FIELD_HEIGHT//2)
ball       = Ball(320, FIELD_HEIGHT//2)

stats = {'saves': 0, 'goals': 0, 'shots': 0}

## 11. Main Game Loop
Runs the simulation at 60 FPS. Close the Pygame window to exit.

In [18]:
running       = True
shoot_pressed = False

while running:
    keys    = pygame.key.get_pressed()
    mouse_x, mouse_y = pygame.mouse.get_pos()

    # ── Events ───────────────────────────────────────────────────────────────
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE and attacker.has_ball:
                gcx = FIELD_WIDTH - 50
                gcy = FIELD_HEIGHT // 2
                tx  = gcx*0.7 + mouse_x*0.3
                ty  = gcy*0.7 + mouse_y*0.3
                ball.shoot(tx, ty, attacker)
                stats['shots'] += 1
                shoot_pressed   = True
            elif event.key == pygame.K_r:
                ball.reset_to_attacker(attacker)

        elif event.type == pygame.KEYUP:
            if event.key == pygame.K_SPACE:
                shoot_pressed = False

    # ── Update ───────────────────────────────────────────────────────────────
    attacker.update(keys)
    ball.update(attacker)
    goalkeeper.update(attacker, ball, shoot_pressed)

    # Save
    if ball.is_shot and goalkeeper.check_save(ball):
        stats['saves'] += 1
        ball.reset_to_attacker(attacker)

    # Goal
    if ball.is_shot and goal.check_goal(ball):
        stats['goals'] += 1
        ball.reset_to_attacker(attacker)
        attacker.x = 300;  attacker.y = FIELD_HEIGHT//2

    # Auto-reset when ball leaves field
    if (ball.x<-50 or ball.x>FIELD_WIDTH+50 or
        ball.y<-50 or ball.y>FIELD_HEIGHT+50 or
        (not ball.is_shot and not attacker.has_ball
         and abs(ball.dx)<0.1 and abs(ball.dy)<0.1)):
        ball.reset_to_attacker(attacker)

    # ── Draw ─────────────────────────────────────────────────────────────────
    pitch.draw(screen)
    goal.draw(screen)
    attacker.draw(screen)
    ball.draw(screen)
    goalkeeper.draw(screen)
    draw_ui(goalkeeper, attacker, ball, stats)

    pygame.display.flip()
    clock.tick(60)

pygame.quit()